In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col
spark = SparkSession.builder.appName("Silver order reviews validation").getOrCreate()

c:\Users\LENOVO\Desktop\AdvancedPySparkPipeline\.venv\Lib\site-packages\pyspark\testing\utils.py:127: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()


In [2]:
bronze_order_reviews = spark.read.parquet("../data/bronze/reviews")
silver_order_reviews = spark.read.parquet("../data/silver/reviews")

In [3]:
bronze_order_reviews.count()

99224

In [4]:
silver_order_reviews.count()

99224

In [5]:
bronze_order_reviews.printSchema()

root
 |-- review_id: string (nullable = true)
 |-- order_id: string (nullable = true)
 |-- review_score: integer (nullable = true)
 |-- review_comment_title: string (nullable = true)
 |-- review_comment_message: string (nullable = true)
 |-- review_creation_date: timestamp (nullable = true)
 |-- review_answer_timestamp: timestamp (nullable = true)



In [6]:
silver_order_reviews.printSchema()

root
 |-- review_id: string (nullable = true)
 |-- order_id: string (nullable = true)
 |-- review_score: integer (nullable = true)
 |-- review_comment_title: string (nullable = true)
 |-- review_comment_message: string (nullable = true)
 |-- review_creation_date: timestamp (nullable = true)
 |-- review_answer_timestamp: timestamp (nullable = true)



In [8]:
from pyspark.sql.functions import count,when,col

In [9]:
silver_order_reviews.select([
    count(when(col(c).isNull(),c)).alias(c)
    for c in silver_order_reviews.columns
]).show(vertical=True)

-RECORD 0------------------------
 review_id               | 0     
 order_id                | 0     
 review_score            | 0     
 review_comment_title    | 87656 
 review_comment_message  | 58247 
 review_creation_date    | 0     
 review_answer_timestamp | 0     



In [10]:
duplicate_silver_order_reviews = silver_order_reviews.groupBy("review_id").count().filter(col("count")>1)
duplicate_silver_order_reviews.show()
print("Duplicate Silver Order Reviews:",duplicate_silver_order_reviews.count())

+--------------------+-----+
|           review_id|count|
+--------------------+-----+
|f144ac19984746532...|    2|
|e4f5fbbbf2fa8f259...|    2|
|868c4827b9ab72723...|    2|
|a8ffb30c74397c7cf...|    2|
|f6e05a3f6dbc3bdb1...|    2|
|c9a406d3076559e35...|    2|
|95cfb15ad3477fec5...|    2|
|dbcd301b59c7e85e6...|    2|
|9a07cb8136a1792e7...|    2|
|dd43a798c9e8baf7b...|    2|
|ef60daa08e71f279b...|    2|
|4812973f7fb59a488...|    2|
|f45e3373e177f5694...|    2|
|9d7b240dff5539983...|    2|
|4e9a141db068cf685...|    2|
|c1dd7c6a2154caaf1...|    2|
|bee3c76b6017ec9de...|    2|
|1961cafe1d1cbd4cc...|    2|
|d8a302244615111a0...|    2|
|fde5986d35c89aa1b...|    2|
+--------------------+-----+
only showing top 20 rows
Duplicate Silver Order Reviews: 789


In [11]:
silver_order_reviews.groupBy(silver_order_reviews.columns).count().filter("count > 1").show()

+---------+--------+------------+--------------------+----------------------+--------------------+-----------------------+-----+
|review_id|order_id|review_score|review_comment_title|review_comment_message|review_creation_date|review_answer_timestamp|count|
+---------+--------+------------+--------------------+----------------------+--------------------+-----------------------+-----+
+---------+--------+------------+--------------------+----------------------+--------------------+-----------------------+-----+



In [12]:
silver_order_reviews.select("review_score").distinct().show()

+------------+
|review_score|
+------------+
|           1|
|           3|
|           5|
|           4|
|           2|
+------------+

